# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bajwaycodes/Kashif-Working-Repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/bajwaycodes/Kashif-Working-Repo"
REPO_DIR = "Kashif-Working-Repo"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring. The real question — "which pages
should an editor review first?" — is a ranking problem, not a classification problem.
The framing skill's task-type table maps "which ones first?" questions to
ranking/scoring, with a priority score as the output and precision@K as the metric.
That's exactly this lane.

Underneath the ranking, there's a classification-shaped piece: to build a score for
each page, I need some estimate of "is this page likely declining" — which is a
binary signal. So a classifier's output (a probability) is a natural ingredient for
the score. But the thing that actually gets evaluated and used is the ranking's
quality at the top of the list — not raw classification accuracy across all 30,000
pages, since an editor will only ever look at the top 20 or 50

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target used: `is_declining_label`**

This comes from `trend_direction == "down"`, which itself is computed from `trend_pct`
— a percentage change calculated over the current trailing-90-day window.

Applying the framing skill's rule directly: is this label *observed* (something that
happened, measured after the fact) or *defined* (a bucket someone's rule created)? It's
defined — `trend_direction` is a threshold applied to `trend_pct`, not an outcome
observed in a later time period. That makes `is_declining_label` a **proxy label**,
not a true target.

I'm using it anyway for this framing exercise because it's what the starter dataset
ships with, but I want to name the honest limitation: a model trained on this proxy
is really learning to reproduce whatever rule defines "down," not learning to predict
a genuine future event. The lane guide itself flags this same label as a beginner
starting point, and names the stronger version explicitly: prior 90 days of features
predicting decline over the *next* 30 days — an outcome that hasn't happened yet at
prediction time. Building that version would require the warehouse's daily fact table
rather than this single-snapshot starter CSV.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@K** (specifically Precision@20 and Precision@50).

An editor doesn't review the whole ranked list — they work through a fixed-size chunk,
whatever their week allows. Precision@K asks exactly the question that matches that
workflow: "of the top K pages I told someone to check, how many actually turned out to
be declining?" That ties directly to the cost of a wrong call from Week 1 — a low
Precision@K means wasted editor time on pages that didn't need review.

"Good" here means beating the baseline hand rule's Precision@K, since that's the
honest thing to beat first, not an arbitrary target like "90% precision." Below I
compute Precision@20 and Precision@50 for the hand rule so there's a concrete number
on record before any model gets built.

I'm not using plain accuracy, because with a ~54% declining rate, a model could score
high accuracy while still producing a useless ranking — accuracy doesn't care about
*order*, and order is the entire point of a review queue. Recall also matters (missing
a real decline is the costlier mistake), so I'll track it alongside Precision@K, but
Precision@K stays primary since it's what determines whether the top of the list is
actually worth an editor's time.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule Precision@20: 0.900
Hand rule Precision@50: 0.680


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one content page, for one client, summarized over a
trailing 90-day window.**

The grain is `content_id`, scoped within `client_id` — not one row per day, not one
row per query, not one row per client. Each page appears exactly once in this
dataset, carrying its own 90-day rollup of impressions, position, CTR, and so on.

In [4]:
unit_cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
             "avg_position", "ctr", "content_age_days", "trend_direction"]
df[unit_cols].head(5)

,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,141,down
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,263,down


In [5]:
dupe_check = df.groupby("content_id").size()
print("Max rows per content_id:", dupe_check.max())        # should print 1
print("Unique content_ids:", df["content_id"].nunique(), "out of", len(df), "rows")

Max rows per content_id: 1
Unique content_ids: 30000 out of 30000 rows


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a fixed rule here**

I tested this directly in Week 1, not just in theory. A one-line hand rule —
`stale (days_since_last_update >= 180) AND visible (impressions_90d >= 500)`, ranked
by impressions — scored Precision@20 = 0.900 and Precision@50 = 0.680. A depth-3
decision tree, given the same signals, scored Precision@20 = 0.700 and
Precision@50 = 0.720.

Read together, those numbers tell a specific story: the hand rule is excellent right
at the very top of the list, but runs out of nuance deeper in — by 50 candidates, a
model that can weigh several signals against each other starts finding pages the
single if-statement misses.

What makes the pattern too messy for an if-statement is that "worth reviewing" isn't
one condition, it's several interacting ones: a stale-but-invisible page and a
stale-and-visible page behave differently; a page with high impressions but strong
CTR looks fine even if it's old; a fresh page with declining position might need
attention despite passing the "stale" test entirely. A hand rule can only test one
fixed combination of thresholds at a time. A model can learn where those thresholds
should actually sit, and how signals trade off against each other, directly from data.

I'm not overclaiming this, though. When I ran a proper client-holdout validation
(grouping by `client_id` so no client's pages leaked between train and test), the
tree's advantage wasn't stable — across different random seeds, the tree sometimes
beat the hand rule and sometimes lost, because with only 32 clients and a few very
large ones, which clients land in the test set can swing the result substantially.
So the honest claim is: ML is a strong *candidate* here, worth building and validating
properly — not a settled win from a single in-sample comparison.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.